In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model


In [2]:
# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, input_dim, emb_dim, hidden_dim):
      super(Encoder, self).__init__()
      self.embedding = layers.Embedding(input_dim, emb_dim)
      self.lstm = layers.LSTM(hidden_dim, return_state=True)

    def call(self, src):
      embedded = self.embedding(src)
      outputs, hidden, cell = self.lstm(embedded)
      return  hidden, cell

In [3]:
# Decoder
class Decoder(tf.keras.Model):
  def __init__(self, output_dim, emb_dim, hidden_dim):
    super(Decoder, self).__init__()
    self.embedding = layers.Embedding(output_dim, emb_dim)
    self.lstm = layers.LSTM(hidden_dim,  return_state=True)
    self.fc_out = layers.Dense(output_dim)

  def call(self, input, hidden, cell):
    embedded = self.embedding(input)
    embedded = tf.expand_dims(embedded, 1)
    output, hidden, cell = self.lstm(embedded, initial_state=[hidden, cell])
    prediction = self.fc_out(output)
    return prediction, hidden, cell

In [18]:
## Seq2Seq

class Seq2Seq(tf.keras.Model):
  def __init__(self, encoder, decoder):
    super(Seq2Seq, self).__init__()
    self.encoder = encoder
    self.decoder = decoder

  def call(self, src, trg):
    batch_size = src.shape[0]
    trg_len = trg.shape[1]
    trg_vocab_size = self.decoder.fc_out.units


    outputs=[]

    hidden, cell = self.encoder(src)

    input = trg[:, 0]

    for t in range(1, trg_len):
      print(f"time step {t} : input shape before decoding: {input.shape}")
      output, hidden, cell = self.decoder(input, hidden, cell)


      print(f"time step {t} : output shape from decoder: {output.shape}")
      outputs.append(output)
      input = trg[:, t]

    outputs = tf.stack(outputs, axis=-1)

    return outputs

In [19]:
src= tf.random.uniform([32, 20], maxval=10000, dtype=tf.int32)
trg= tf.random.uniform([32, 20], maxval=10000, dtype=tf.int32)

input_dim = 10000
output_dim = 10000
emb_dim = 256

hidden_dim = 512

encoder = Encoder(input_dim, emb_dim, hidden_dim)
decoder = Decoder(output_dim, emb_dim, hidden_dim)
seq2seqmodel = Seq2Seq(encoder, decoder)

output = seq2seqmodel(src, trg)
print({output.shape})

time step 1 : input shape before decoding: (32,)
time step 1 : output shape from decoder: (32, 10000)
time step 2 : input shape before decoding: (32,)
time step 2 : output shape from decoder: (32, 10000)
time step 3 : input shape before decoding: (32,)
time step 3 : output shape from decoder: (32, 10000)
time step 4 : input shape before decoding: (32,)
time step 4 : output shape from decoder: (32, 10000)
time step 5 : input shape before decoding: (32,)
time step 5 : output shape from decoder: (32, 10000)
time step 6 : input shape before decoding: (32,)
time step 6 : output shape from decoder: (32, 10000)
time step 7 : input shape before decoding: (32,)
time step 7 : output shape from decoder: (32, 10000)
time step 8 : input shape before decoding: (32,)
time step 8 : output shape from decoder: (32, 10000)
time step 9 : input shape before decoding: (32,)
time step 9 : output shape from decoder: (32, 10000)
time step 10 : input shape before decoding: (32,)
time step 10 : output shape from